# FX Multi-ECN Pipeline — Pro Version

This notebook uses the packaged modules in `fx_ecn_pipeline/` to run the full pipeline,
including optional XGBoost / PyTorch models (if available) and a simple executable backtest.


In [ ]:
import os, pandas as pd, numpy as np
from fx_ecn_pipeline import (
    PipelineConfig, load_data, compute_latency_stats, build_calendar_grid_merge_asof,
    build_features_from_grid, build_classification_target, build_regression_target,
    eval_cls_venue, eval_reg_venue, eval_latency_buckets, xgb_available, torch_available,
    hasbrouck_info_share, granger_against_agg, backtest_threshold_strategy
)

cfg = PipelineConfig().ensure_dirs()
print(cfg)

In [ ]:
df = load_data(cfg.data_path)
venues = sorted(df['venue'].unique())
print('Rows:', len(df), 'Venues:', venues)
df.head()

In [ ]:
lat_stats = compute_latency_stats(df)
lat_stats.to_csv(os.path.join(cfg.out_dir, 'latency_stats.csv'), index=False)
lat_stats.head()

In [ ]:
grid_df = build_calendar_grid_merge_asof(df, grid_ms=cfg.grid_ms)
grid_df.to_csv(os.path.join(cfg.out_dir, f'aligned_grid_{cfg.grid_ms}ms.csv'), index=False)
grid_df.head()

In [ ]:
feat_df = build_features_from_grid(grid_df, venues)
feat_df.to_csv(os.path.join(cfg.out_dir, 'features_grid.csv'), index=False)
feat_df.head()

In [ ]:
labeled_cls = build_classification_target(feat_df, grid_ms=cfg.grid_ms, delta_ms=cfg.delta_ms, thr=0.0)
labeled_reg = build_regression_target(feat_df, grid_ms=cfg.grid_ms, delta_ms=cfg.delta_ms)
labeled_cls.to_csv(os.path.join(cfg.out_dir, f'labeled_cls_{cfg.delta_ms}ms.csv'), index=False)
labeled_reg.to_csv(os.path.join(cfg.out_dir, f'labeled_reg_{cfg.delta_ms}ms.csv'), index=False)
print('Class balance:', labeled_cls['target_dir'].value_counts(dropna=False).to_dict())

In [ ]:
cls_res = [eval_cls_venue(labeled_cls, v) for v in venues]
reg_res = [eval_reg_venue(labeled_reg, v) for v in venues]
pd.DataFrame(cls_res), pd.DataFrame(reg_res)

In [ ]:
print('XGBoost available?', xgb_available(), '| Torch available?', torch_available())

In [ ]:
lat_bucket = {v: eval_latency_buckets(df, labeled_cls, v) for v in venues}
lat_bucket

In [ ]:
info_df = hasbrouck_info_share(grid_df, nlags=3)
info_df

In [ ]:
gc = granger_against_agg(grid_df, venues, maxlag=5)
gc

In [ ]:
# quick demo backtest using a simple in-sample logistic reg on the first venue
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
v0 = venues[0]
cols = [c for c in [f'mid_{v0}', f'mid_{v0}_ret_1', f'mid_{v0}_ret_3', f'mid_{v0}_volatility_10', f'mid_{v0}_diff_to_agg', f'depth_{v0}', f'depth_imbalance_{v0}'] if c in labeled_cls.columns]
pack = labeled_cls.dropna(subset=['target_dir'] + (cols if cols else [f'mid_{v0}'])).copy()
X = pack[cols if cols else [f'mid_{v0}']].values; y = pack['target_dir'].values
sc = StandardScaler().fit(X)
clf = LogisticRegression(max_iter=500).fit(sc.transform(X), y)
pack['prob_up'] = clf.predict_proba(sc.transform(X))[:,1]
bt = backtest_threshold_strategy(pack[['grid_time','agg_mid','prob_up']].copy(), entry_th=0.55, exit_th=0.5, fee_bps=cfg.fee_bps, slip_bps=cfg.slip_bps)
bt[['grid_time','pos','pnl','eq_curve']].head()